#### Environment Check


In [ ]:
import sys
print(sys.executable)


#### Setup


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


#### Project Paths


In [ ]:
cwd = Path.cwd()

if (cwd / "code").exists() and (cwd / "data").exists():
    LAB_DIR = cwd
elif (cwd.parent / "code").exists() and (cwd.parent / "data").exists():
    LAB_DIR = cwd.parent
else:
    LAB_DIR = Path("..").resolve()

CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

DATA_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

str(LAB_DIR)


#### Import Course Helpers


In [ ]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data, build_index
from evaluation_utils import RAGWithUsage, calc_total_price, map_progress


#### Load OpenAI Client


In [ ]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"
loaded = load_dotenv(ENV_PATH)
print("Env file:", ENV_PATH)
print("Env loaded:", loaded)
openai_client = OpenAI()
MODEL = "gpt-5.4-mini"


#### Load Ground Truth


In [ ]:
ground_truth_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth = pd.read_csv(ground_truth_path)
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)


#### Load FAQ Documents


In [ ]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

len(documents)


#### Build Search Index


In [ ]:
index = build_index(documents)

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

len(doc_idx)


#### Create RAG Assistant


In [ ]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    model=MODEL,
)


#### Test One RAG Answer


In [ ]:
rec = ground_truth[0]
question = rec["question"]

answer_llm, usage = assistant.rag(question)

print(question)
print()
print(answer_llm)


#### Compare With Original Answer


In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

print(answer_orig)


#### Create RAG Answer Function


In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm, usage = assistant.rag(question)

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": original_doc["answer"],
        "document": doc_id,
    }

    return result, usage


#### Test Function


In [ ]:
answer_record, usage = generate_rag_answer(ground_truth[0])

answer_record


#### Generate RAG Answers


In [ ]:
questions_to_process = ground_truth
# questions_to_process = ground_truth[:20]

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, questions_to_process, generate_rag_answer)


#### Split Answers And Usage


In [ ]:
answers = []
usages = []

for answer_record, usage in results:
    answers.append(answer_record)
    usages.append(usage)

len(answers)


#### Calculate RAG Cost


In [ ]:
total_cost = calc_total_price(usages)

total_cost


#### Save RAG Answers


In [ ]:
df_answers = pd.DataFrame(answers)

output_path = DATA_DIR / "rag-answers-new.csv"
df_answers.to_csv(output_path, index=False)

output_path


#### Save RAG Report


In [ ]:
report_path = REPORTS_DIR / "rag_evaluation.md"

report_lines = [
    "# RAG Answer Generation",
    "",
    f"- Questions processed: {len(df_answers)}",
    f"- Total cost: {total_cost}",
    f"- Output file: {output_path.name}",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path
